In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

from Feature_extraction.feature_extractor import FeatureExtractor
from Embeddings.Models.embedding_generator import EmbeddingGenerator
from Embeddings.Semantics.descriptions import get_descriptions
from Model.model import IP_SAE_MODEL

In [2]:
DATASET_PATH = r"../Data/UiS4ADL/Processed/UiS4ADL_100hz.csv"
ADL_DICT_PATH = '../adl_dict.json'

# Parameters
FS = 100
WINDOW_SECONDS = 4.0
OVERLAP_RATIO = 0.0
METHOD = 'temporal_frequency'

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
DESCRIPTION_TYPE = 'original_fadi'
USE_PROMPT = True

# Experimental Setup
DETERMINISTIC_RANKING = [17, 14, 11, 9, 24, 5, 7, 12, 10, 8, 6, 21, 23, 20, 22, 15, 13, 1, 19, 4, 16, 2, 18, 3]
ALL_CLASSES = sorted(DETERMINISTIC_RANKING)
K = 6

In [3]:
embedding_generator = EmbeddingGenerator()
embeddings = embedding_generator.load_embeddings(
    model_name=EMBEDDING_MODEL, desc_type=DESCRIPTION_TYPE, 
    use_prompt=USE_PROMPT, dataset='fadi'
)

# Map class labels to embedding indices
activity_labels = sorted(get_descriptions(DESCRIPTION_TYPE).keys())
class_to_embedding_idx = {label: idx for idx, label in enumerate(activity_labels)}

max_class_id = max(ALL_CLASSES)
all_embeddings_indexed = np.zeros((max_class_id + 1, embeddings.shape[1]))
for cls, idx in class_to_embedding_idx.items():
    if cls in ALL_CLASSES:
        all_embeddings_indexed[cls] = embeddings[idx]

# Initialize Feature Extractor
feature_extractor = FeatureExtractor()
data_proc = pd.read_csv(DATASET_PATH)
sensor_cols = [col for col in data_proc.columns if col not in ['timestamp', 'adl', 'session', 'subject', 'fileID']]

X_proc, y_proc, fileIDs, subjects = feature_extractor.extract_features(
    data=data_proc, method=METHOD, sensor_columns=sensor_cols,
    window_seconds=WINDOW_SECONDS, overlap_ratio=OVERLAP_RATIO, fs=FS, strategy="retain_short"
)

y_proc = np.array(y_proc)

Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_original_fadi_prompt_fadi.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)

Extracting features: temporal_frequency
  Window: 4.0s
  Overlap: 0.0 (400 stride)
  Sampling rate: 100 Hz

Creating windows per fileID...
  Found 1445 files that are too short (30.94% of the available files)
  Created 20103 windows

Extracting features using method: temporal_frequency...
  Processing window 0/20103
                    1000/20103
                    2000/20103
                    3000/20103
                    4000/20103
                    5000/20103
                    6000/20103
                    7000/20103
                    8000/20103
                    9000/20103
                    10000/20103
                    11000/20103
                    12000/20103
                    13000/20103
                    14000/20103
                    15000/20103
                    16000/20103
                    17000/20103
               

In [4]:
def tune_lambda_loco(X_train, y_train, embeddings_all, lambda_grid, n_val_classes=7, n_iterations=35, random_seed=22):
    seen_classes = np.unique(y_train)
    lambda_scores = {l: [] for l in lambda_grid}
    rng = np.random.default_rng(random_seed)

    for l_val in lambda_grid:
        for _ in range(n_iterations):
            val_classes = rng.choice(seen_classes, n_val_classes, replace=False)
            train_classes = [c for c in seen_classes if c not in val_classes]

            cv_train_mask = np.isin(y_train, train_classes)
            cv_val_mask = np.isin(y_train, val_classes)

            X_cv_train, y_cv_train = X_train[cv_train_mask], y_train[cv_train_mask]
            X_cv_val, y_cv_val = X_train[cv_val_mask], y_train[cv_val_mask]

            model = IP_SAE_MODEL(lambda_reg=l_val, scale_features=True)
            model.fit(X_cv_train, y_cv_train, embeddings_all, verbose=0)

            val_class_embeddings = embeddings_all[val_classes]
            y_pred = model.predict_zsl(X_cv_val, val_class_embeddings, val_classes, n_runs=35)

            score = balanced_accuracy_score(y_cv_val, y_pred)
            lambda_scores[l_val].append(score)

        avg_score = np.mean(lambda_scores[l_val])
        std_score = np.std(lambda_scores[l_val])
        print(f"Lambda: {l_val:<8} | Mean Validation Accuracy: {avg_score:.4f} ± {std_score:.4f}")

    best_lambda = max(lambda_scores, key=lambda l: np.mean(lambda_scores[l]))
    return best_lambda, lambda_scores

In [5]:
unseen_classes = DETERMINISTIC_RANKING[:K]
seen_classes = [c for c in ALL_CLASSES if c not in unseen_classes]

train_mask = np.isin(y_proc, seen_classes)
X_train_main = X_proc[train_mask]
y_train_main = y_proc[train_mask]

test_mask = np.isin(y_proc, unseen_classes)
X_test_main = X_proc[test_mask]
y_test_main = y_proc[test_mask]

lambda_grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]

# Run Tuning on Seen Classes only
best_l, all_cv_scores = tune_lambda_loco(
    X_train_main, y_train_main, 
    all_embeddings_indexed,
    lambda_grid
)

print(f"\nOptimal Lambda found: {best_l}")

# Final Evaluation on the true Unseen Classes
print(f"\nFinal Evaluation on Test Set (K={K}) using Lambda={best_l}...")
final_model = IP_SAE_MODEL(lambda_reg=best_l,  scale_features=True)
final_model.fit(X_train_main, y_train_main, all_embeddings_indexed, verbose=0)

unseen_emb_indices = [class_to_embedding_idx[c] for c in unseen_classes]
y_pred_final = final_model.predict_zsl(
    X_test_main, embeddings[unseen_emb_indices], unseen_classes, n_runs=35
)

final_acc = balanced_accuracy_score(y_test_main, y_pred_final)
print(f"Final balanced accuracy (K={K}): {final_acc*100:.2f}%")

Lambda: 1e-05    | Mean Validation Accuracy: 0.2046 ± 0.0512
Lambda: 0.0001   | Mean Validation Accuracy: 0.2047 ± 0.0525
Lambda: 0.001    | Mean Validation Accuracy: 0.2242 ± 0.0630
Lambda: 0.01     | Mean Validation Accuracy: 0.1893 ± 0.0480
Lambda: 0.1      | Mean Validation Accuracy: 0.1581 ± 0.0256
Lambda: 1.0      | Mean Validation Accuracy: 0.1460 ± 0.0064
Lambda: 10.0     | Mean Validation Accuracy: 0.1426 ± 0.0030

Optimal Lambda found: 0.001

Final Evaluation on Test Set (K=6) using Lambda=0.001...
Final balanced accuracy (K=6): 70.76%
